# Structural Probes for Spanish Syntax — Colab runner

Runs the full experimental pipeline on a Colab GPU. Auto-detects the GPU and tunes batch sizes — works fine on T4 (free) but really shines on L4 / A100 / G4 / H100.

**Expected total runtime by GPU (rough):**

| GPU | VRAM | Embeddings | Layer sweep | Lin + iso | TOTAL |
|---|---|---|---|---|---|
| T4 (free) | 16 GB | 10-15 min | 30-50 min | 10-15 min | ~50-80 min |
| L4 (Pro) | 24 GB | 5-10 min | 15-30 min | 5-10 min | ~25-50 min |
| A100 (Pro+) | 40 GB | 3-6 min | 8-15 min | 3-6 min | ~15-30 min |
| **G4 / RTX PRO 6000 Blackwell** | **96 GB** | **2-4 min** | **5-10 min** | **2-4 min** | **~10-20 min** |
| H100 (Pro+) | 80 GB | 2-5 min | 6-12 min | 2-5 min | ~10-25 min |

## Before running
1. `Runtime → Change runtime type → G4 GPU` (or any other available).
2. Decide how to get the project into Colab — see Cell 1.

## Cell 1 — Get the project into Colab

**Option A (recommended if you have GitHub):** set `REPO_URL` below.

**Option B (no GitHub):** leave `REPO_URL = ''`; you'll be prompted to upload a `.zip`. Make the zip on Windows with the included `make_colab_zip.ps1` script.

In [ ]:
import os, subprocess, zipfile

REPO_URL = ''  # e.g. 'https://github.com/marcoslahozy14/Structural-Probes-for-Spanish-Syntax.git'
REPO_DIR = 'Structural-Probes-for-Spanish-Syntax'

if REPO_URL:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    from google.colab import files
    print('Upload a .zip of the project (must contain scripts/, data/, es_ancora.yaml ...)')
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    # Use Python's zipfile rather than the unix `unzip` command, so that
    # zips produced by Windows PowerShell's Compress-Archive (which can use
    # backslash separators) extract reliably.
    with zipfile.ZipFile(zip_name, 'r') as zf:
        members = zf.namelist()
        print(f'Zip contains {len(members)} entries. First 5: {members[:5]}')
        zf.extractall('.')
    candidates = [d for d in os.listdir('.') if os.path.isdir(d) and os.path.isdir(os.path.join(d, 'scripts'))]
    if candidates:
        os.chdir(candidates[0])
    elif not os.path.isdir('scripts'):
        raise RuntimeError('Could not find a folder containing scripts/. Inspect the zip contents printed above.')

print('cwd:', os.getcwd())
print('contents:', sorted(x for x in os.listdir('.') if not x.startswith('.'))[:20])

## Cell 2 — Install dependencies and detect GPU

Auto-tunes `EMBED_BATCH` and `PROBE_BATCH` based on the GPU's VRAM (more robust than name-matching). Recognises T4, L4, V100, A100, **G4 (RTX PRO 6000 Blackwell, 96 GB)**, and H100.

In [ ]:
!pip install -q transformers h5py scipy PyYAML tqdm seaborn
import torch, transformers, h5py, scipy, yaml, numpy, subprocess
print('torch        :', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('transformers :', transformers.__version__)
print('numpy        :', numpy.__version__)

if not torch.cuda.is_available():
    EMBED_BATCH = 8; PROBE_BATCH = 32
    print('\nWARNING: no GPU detected. Runtime → Change runtime type → GPU and re-run from Cell 1.')
else:
    print('\n--- nvidia-smi ---')
    print(subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version,compute_cap',
         '--format=csv'], text=True))

    name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU       : {name}')
    print(f'VRAM (GB) : {vram_gb:.1f}')

    # VRAM-based heuristic — works for any GPU including G4 / RTX PRO 6000
    if vram_gb >= 70:
        # H100 80GB, G4 / RTX PRO 6000 Blackwell 96GB
        EMBED_BATCH = 256; PROBE_BATCH = 256
    elif vram_gb >= 35:
        # A100 40GB
        EMBED_BATCH = 128; PROBE_BATCH = 128
    elif vram_gb >= 20:
        # L4 24GB
        EMBED_BATCH = 96;  PROBE_BATCH = 96
    elif vram_gb >= 14:
        # T4 16GB, V100 16GB
        EMBED_BATCH = 64;  PROBE_BATCH = 64
    else:
        EMBED_BATCH = 32;  PROBE_BATCH = 32

    print(f'\nBatch sizes auto-tuned by VRAM:')
    print(f'  EMBED_BATCH (mBERT inference) : {EMBED_BATCH}')
    print(f'  PROBE_BATCH (probe training)  : {PROBE_BATCH}')

## Cell 3 — Patch es_ancora.yaml with the tuned probe batch size

In [ ]:
import yaml
with open('es_ancora.yaml') as f:
    cfg = yaml.safe_load(f)
old = cfg['dataset']['batch_size']
cfg['dataset']['batch_size'] = PROBE_BATCH
with open('es_ancora.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(f'Patched es_ancora.yaml: dataset.batch_size {old} -> {PROBE_BATCH}')

## Cell 4 — Verify (or download) the AnCora CoNLL-U files

In [ ]:
import os
DATA = 'data/es_ancora'
needed = [f'{DATA}/es_ancora-ud-{s}.conllu' for s in ('train', 'dev', 'test')]
if all(os.path.exists(p) for p in needed):
    for p in needed:
        print(f'  {p}: {os.path.getsize(p) / 1e6:.1f} MB')
else:
    print('Missing CoNLL-U files; downloading UD_Spanish-AnCora ...')
    os.makedirs(DATA, exist_ok=True)
    !git clone -q --depth 1 https://github.com/UniversalDependencies/UD_Spanish-AnCora.git /tmp/UD_Spanish-AnCora
    for s in ('train', 'dev', 'test'):
        !cp /tmp/UD_Spanish-AnCora/es_ancora-ud-{s}.conllu {DATA}/es_ancora-ud-{s}.conllu
        print(f'  copied {DATA}/es_ancora-ud-{s}.conllu')

## Cell 5 — Step 1: CoNLL-U → text

Drops contraction range rows and enhanced empty nodes. ~seconds.

In [ ]:
for split in ('train', 'dev', 'test'):
    !python -m scripts.conllu_to_text data/es_ancora/es_ancora-ud-{split}.conllu data/es_ancora/es_ancora-ud-{split}.txt

## Cell 6 — Step 2: Generate mBERT embeddings (all 13 layers)

First call downloads `bert-base-multilingual-cased` (~700 MB; one-off). Uses `EMBED_BATCH` from Cell 2.

In [ ]:
for split in ('train', 'dev', 'test'):
    print(f'\n--- Encoding {split} (batch={EMBED_BATCH}) ---')
    !python -m scripts.generate_embeddings \
        data/es_ancora/es_ancora-ud-{split}.txt \
        data/es_ancora/es_ancora-ud-{split}.hdf5 \
        --layers all --aggregation mean --batch-size {EMBED_BATCH}

In [ ]:
import os
for f in sorted(os.listdir('data/es_ancora')):
    if f.endswith('.hdf5'):
        print(f'  data/es_ancora/{f}: {os.path.getsize(f"data/es_ancora/{f}") / 1e6:.1f} MB')
print()
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

## Cell 7 — Step 3: Layer sweep (linear probe over all 13 layers)

In [ ]:
!python -m scripts.run_layer_sweep es_ancora.yaml --layers 0 1 2 3 4 5 6 7 8 9 10 11 12 --seed 0

In [ ]:
import csv, glob
csvs = sorted(glob.glob('results/es_ancora/layersweep-*/sweep.csv'))
assert csvs, 'No sweep.csv found — did the sweep finish?'
csv_path = csvs[-1]
print('Reading', csv_path)
rows = list(csv.DictReader(open(csv_path)))
for r in rows:
    print(r)
best = max(rows, key=lambda r: float(r['dev_spearman_5_50'] or 0))
BEST_LAYER = int(best['layer'])
print(f'\n>>> BEST LAYER = {BEST_LAYER}'
      f"   (dev_spearman={best['dev_spearman_5_50']}, dev_uuas={best['dev_uuas']})")

## Cell 8 — Step 4: Linear vs. isometric probe at the best layer

In [ ]:
import yaml, copy, subprocess

with open('es_ancora.yaml') as f:
    base = yaml.safe_load(f)
base['model']['model_layer'] = BEST_LAYER
base['dataset']['batch_size'] = PROBE_BATCH

for kind in ('linear', 'isometric'):
    cfg = copy.deepcopy(base)
    cfg['probe']['isometric'] = (kind == 'isometric')
    out_yaml = f'es_ancora_{kind}.yaml'
    with open(out_yaml, 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    print(f'\n=== Training {kind} probe at layer {BEST_LAYER} (batch={PROBE_BATCH}) ===')
    subprocess.run(['python', '-m', 'scripts.run_experiment', out_yaml, '--seed', '0'], check=True)

## Cell 9 — Step 5: Geometric diagnostic (condition number)

In [ ]:
import glob, os
lin_dirs = sorted(glob.glob('results/es_ancora/BERT-disk-parse-distance-*/'), key=os.path.getmtime)
iso_dirs = sorted(glob.glob('results/es_ancora/iso/BERT-disk-parse-distance-*/'), key=os.path.getmtime)
lin = lin_dirs[-1] if lin_dirs else None
iso = iso_dirs[-1] if iso_dirs else None
print('Linear probe dir   :', lin)
print('Isometric probe dir:', iso)

In [ ]:
if lin:
    !python -m scripts.calc_condition_number {lin}predictor.params

In [ ]:
if iso:
    !python -m scripts.calc_condition_number {iso}predictor.params --config es_ancora_isometric.yaml

## Cell 10 — Step 6: Plot Spearman / UUAS vs. layer

In [ ]:
import matplotlib.pyplot as plt

layers = [int(r['layer']) for r in rows]
spr = [float(r['dev_spearman_5_50'] or float('nan')) for r in rows]
uuas = [float(r['dev_uuas'] or float('nan')) for r in rows]

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(layers, spr, 'o-', color='C0', label='Spearman 5–50 (dev)')
ax1.set_xlabel('mBERT layer')
ax1.set_ylabel('Spearman ρ', color='C0')
ax1.tick_params(axis='y', labelcolor='C0')
ax1.axvline(BEST_LAYER, color='gray', linestyle='--', alpha=0.5,
            label=f'best = layer {BEST_LAYER}')
ax2 = ax1.twinx()
ax2.plot(layers, uuas, 's--', color='C1', label='UUAS (dev)')
ax2.set_ylabel('UUAS', color='C1')
ax2.tick_params(axis='y', labelcolor='C1')
fig.suptitle('Structural probe over mBERT layers — UD_Spanish-AnCora dev')
fig.tight_layout()
plt.savefig('results/layer_curve.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved results/layer_curve.png')

## Cell 11 — Final summary in chat

In [ ]:
def read1(p):
    try:
        return open(p).readline().strip().split('\t')[0]
    except FileNotFoundError:
        return 'n/a'

print('=== HEADLINE NUMBERS ===\n')
print(f'Best layer          : {BEST_LAYER}\n')
print(f'LINEAR probe (layer {BEST_LAYER}):')
print(f'  dev UUAS          : {read1(lin + "dev.uuas")}')
print(f'  dev Spearman 5-50 : {read1(lin + "dev.spearmanr-5_50-mean")}')
print()
print(f'ISOMETRIC probe (layer {BEST_LAYER}):')
print(f'  dev UUAS          : {read1(iso + "dev.uuas")}')
print(f'  dev Spearman 5-50 : {read1(iso + "dev.spearmanr-5_50-mean")}')
print()
print('Files to look at:')
print(f'  - {csv_path}                     (sweep CSV)')
print(f'  - results/layer_curve.png         (the slide figure)')
print(f'  - {lin}predictor.params           (linear probe weights)')
print(f'  - {iso}predictor.params           (isometric probe weights)')

## Cell 12 — Zip and download results

Excludes the heavy `*.hdf5` embeddings. Triggers a download of `results.zip` to your local Downloads folder.

In [ ]:
!zip -qr results.zip results/ es_ancora_linear.yaml es_ancora_isometric.yaml -x '*.hdf5'
from google.colab import files
print('Downloading results.zip ...')
files.download('results.zip')

## Done

Unzip `results.zip` into `results/colab/` on your local machine. You'll have:

- `sweep.csv` — Spearman/UUAS por capa
- `layer_curve.png` — el gráfico estrella
- Carpetas `BERT-disk-...` y `iso/BERT-disk-...` con `predictor.params` y métricas dev de lineal e isométrico
- `es_ancora_linear.yaml` y `es_ancora_isometric.yaml` — los configs efectivos usados